# CUDA-zkML / PCANI — Protocol v2 Final NVIDIA Validation (2026)

This notebook performs the publication-oriented PCANI v2 validation path. It builds the hardened CUDA/Groth16 code, runs regression gates, measures all candidate paths on a real NVIDIA GPU, calibrates using selected-proof cost semantics, verifies representative single-proof route certificates, and exports `PCANI_final_results.zip`.

The protocol-v1 proof-chain experiment is retained in the repository as historical evidence, but protocol v2 is the default in this notebook.


In [ ]:
# 1) Confirm NVIDIA/CUDA runtime
import os, subprocess, sys, json, pathlib
print(subprocess.check_output(['nvidia-smi'], text=True))
print(subprocess.check_output(['nvcc','--version'], text=True))


In [ ]:
# 2) Upload the supplied CUDA-zkML-PCANI-2026-Colab-Ready.zip
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload exactly one project ZIP; got: {zip_names}')
zip_name = zip_names[0]
extract_root = Path('/content/pcani_src')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_root)
repos = [p.parent for p in extract_root.rglob('CMakeLists.txt') if (p.parent/'python'/'pcani_colab_experiment.py').exists()]
if len(repos) != 1:
    raise RuntimeError(f'Could not identify a unique repository root: {repos}')
REPO = repos[0]
os.chdir(REPO)
print('Repository:', REPO)
print('Version:', (REPO/'VERSION').read_text().strip())


In [ ]:
# 3) Install dependencies needed by the final experiment
!apt-get -qq update
!apt-get -qq install -y cmake build-essential > /dev/null
!python -m pip -q install --upgrade pip
!python -m pip -q install -r requirements.txt
!python -m pip -q install py-ecc
print('Dependencies ready.')


In [ ]:
# 4) Clean RELEASE build. Debug/fast benchmark shortcuts are explicitly OFF.
import shutil, os, subprocess
build = REPO/'build'
if build.exists():
    shutil.rmtree(build)
subprocess.run([
    'cmake','-S',str(REPO),'-B',str(build),
    '-DCMAKE_BUILD_TYPE=Release',
    '-DCMAKE_CUDA_ARCHITECTURES=75',
    '-DZKML_FAST_DEBUG_LOOP=OFF',
    '-DZKML_FAST_ITERATION=OFF',
    '-DZKML_ULTRA_FAST_COMPILE=OFF',
    '-DZKML_NVCC_THREADS=1',
    '-DZKML_NVCC_SPLIT_COMPILE_THREADS=1',
], check=True)
subprocess.run(['cmake','--build',str(build),'-j2'], check=True)
print('Release build complete.')


In [ ]:
# 5) CPU/Python + CUDA/CTest regression gates
import subprocess, os
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO/'python')
subprocess.run([sys.executable,'-m','pytest','-q'], cwd=REPO, env=env, check=True)
subprocess.run(['ctest','--test-dir',str(REPO/'build'),'--output-on-failure'], cwd=REPO, check=True)
print('All available regression gates passed.')


In [ ]:
# 6) Run the real held-out Digits + measured Groth16 PCANI protocol-v2 experiment.
#    Candidate paths: 16,48,96,160 hidden units.
#    5 repeated proofs/path; zero calibration loss allowed.
#    Held-out acceptance tolerance is predeclared at 1 percentage point.
import subprocess, os
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO/'python')
subprocess.run([
    sys.executable, str(REPO/'python'/'pcani_colab_experiment.py'),
    '--build-dir','build',
    '--benchmark-dir','benchmarks/pcani_digits',
    '--results-dir','results/pcani_colab_final',
    '--widths','16,48,96,160',
    '--repeats','5',
    '--max-accuracy-drop','0.0',
    '--heldout-accuracy-drop-limit','0.01',
    '--max-route-bundles','4',
    '--protocol-version','2',
], cwd=REPO, env=env, check=True)


In [ ]:
# 7) Show the measured protocol-v2 result summary. Do not edit these numbers manually.
import json
summary = json.loads((REPO/'results'/'pcani_colab_final'/'run_summary.json').read_text())
final = summary['pcani']['test']
print(json.dumps({
    'protocol_version': summary['protocol_version'],
    'adaptive_accuracy': final['accuracy'],
    'full_path_accuracy': final['full_path_accuracy'],
    'accuracy_drop': final['accuracy_drop'],
    'expected_selected_proof_cost_ms': final['expected_proof_cost'],
    'static_full_proof_cost_ms': final['full_path_cost'],
    'proof_cost_savings': final['proof_cost_savings'],
    'route_counts': final['route_counts'],
    'verified_route_evidence': len(summary['route_evidence_reports']),
}, indent=2))

if summary['hypothesis_supported_on_this_run']:
    print('\nRESULT: PCANI v2 quality/proof-cost hypothesis supported on this benchmark/run.')
else:
    print('\nRESULT: PCANI v2 did not meet the predeclared Pareto criterion on this run. Return the ZIP anyway; do not hide the result.')


In [ ]:
# 8) Download the only file you need to send back.
from google.colab import files
result_zip = REPO/'PCANI_final_results.zip'
if not result_zip.exists():
    raise FileNotFoundError(result_zip)
print('Result ZIP:', result_zip, 'size=', result_zip.stat().st_size, 'bytes')
files.download(str(result_zip))
